# Assignment No. 2 — Data Cleaning and Preprocessing

## Titanic: Machine Learning from Disaster

**Student Name:** Haziq Ali  
**Course:** Data Science  
**Assignment:** Data Cleaning and Preprocessing  

### Dataset Source
[Kaggle — Titanic: Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic)

The Titanic dataset is a beginner-friendly dataset containing passenger information and a target variable showing whether each passenger survived. The Kaggle training dataset contains **891 rows and 12 columns**. The objective is to understand and preprocess the passenger data so it can be used to predict survival.

> **Note:** Download `train.csv` from the Kaggle dataset and upload it to this Colab notebook when prompted in the first code cell.


## 1. Import Libraries and Load the Dataset

We use Pandas for data handling, NumPy for numerical operations, Matplotlib and Seaborn for simple visualizations, and Scikit-learn for preprocessing.

The code below allows the notebook to work in Google Colab. It asks the user to upload the Kaggle `train.csv` file.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

uploaded = files.upload()

df = pd.read_csv("train.csv")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()


### Dataset Description

The main columns are:

- **PassengerId:** Unique passenger ID.
- **Survived:** Target variable; 0 = did not survive, 1 = survived.
- **Pclass:** Passenger class (1st, 2nd, or 3rd).
- **Name:** Passenger's name.
- **Sex:** Passenger's sex.
- **Age:** Passenger age.
- **SibSp:** Number of siblings/spouses aboard.
- **Parch:** Number of parents/children aboard.
- **Ticket:** Ticket number.
- **Fare:** Ticket fare.
- **Cabin:** Cabin number.
- **Embarked:** Port of embarkation.

The dataset contains both numerical and categorical variables, making it suitable for demonstrating data cleaning and preprocessing.


In [ ]:
# Basic information
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nBasic statistics:")
display(df.describe())


## 2. Data Cleaning

### 2.1 Identify Missing Values

First, we check how many missing values exist in each column.

`Age`, `Cabin`, and `Embarked` contain missing values. Instead of blindly deleting rows, we select a suitable method for each column.


In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_percent = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_table = pd.DataFrame({
    "Missing Values": missing,
    "Percentage": missing_percent.round(2)
})

display(missing_table)


### Missing-value decisions

- **Age:** Fill missing values with the median. Age is numerical and the median is less affected by extreme values than the mean.
- **Embarked:** Fill missing values with the mode because it is a categorical variable and only a few values are missing.
- **Cabin:** Drop the original column because a very large proportion of cabin values are missing. Keeping the raw column would add too much missing information. We will instead create a simpler `CabinKnown` feature before dropping it.


In [ ]:
# Create a useful feature before dropping Cabin
df["CabinKnown"] = df["Cabin"].notna().astype(int)

# Fill Age using median
df["Age"] = df["Age"].fillna(df["Age"].median())

# Fill Embarked using mode
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Drop Cabin because most values are missing
df = df.drop(columns=["Cabin"])

print("Missing values after handling:")
print(df.isnull().sum())


## 2.2 Detect and Remove Duplicate Records

Duplicate rows can cause the same observation to be counted more than once. We check for duplicates and remove them if they exist.

`PassengerId` is also a unique identifier, so it can help us verify whether passenger records are duplicated.


In [ ]:
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)


## 2.3 Check Inconsistent Categories

Categorical columns can sometimes contain spelling differences, extra spaces, or unexpected categories. We inspect `Sex` and `Embarked`.

The Titanic data uses consistent categories, but standardizing text with `str.strip()` makes the cleaning process safer.


In [ ]:
# Remove accidental spaces and standardize text
df["Sex"] = df["Sex"].astype(str).str.strip().str.lower()
df["Embarked"] = df["Embarked"].astype(str).str.strip().str.upper()

print("Sex categories:", df["Sex"].unique())
print("Embarked categories:", df["Embarked"].unique())


## 2.4 Correct Data Types

The columns should have appropriate data types. `Survived`, `Pclass`, `SibSp`, `Parch`, and `CabinKnown` represent integer categories/counts. `Age` and `Fare` should be numeric.

We also convert `Sex` and `Embarked` to categorical types because they represent categories rather than continuous measurements.


In [ ]:
df["PassengerId"] = df["PassengerId"].astype(int)
df["Survived"] = df["Survived"].astype(int)
df["Pclass"] = df["Pclass"].astype(int)
df["SibSp"] = df["SibSp"].astype(int)
df["Parch"] = df["Parch"].astype(int)
df["CabinKnown"] = df["CabinKnown"].astype(int)

df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Fare"] = pd.to_numeric(df["Fare"], errors="coerce")

df["Sex"] = df["Sex"].astype("category")
df["Embarked"] = df["Embarked"].astype("category")

print(df.dtypes)


## 2.5 Detect and Handle Outliers

We use the Interquartile Range (IQR) method to identify possible outliers in `Fare`.

A high fare is not automatically an error because expensive first-class tickets are possible. Therefore, we do **not** delete valid passengers just because their fare is high. Instead, we apply a log transformation later to reduce the influence of strong right-skewness.


In [ ]:
Q1 = df["Fare"].quantile(0.25)
Q3 = df["Fare"].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

fare_outliers = df[(df["Fare"] < lower_limit) | (df["Fare"] > upper_limit)]

print("Lower IQR limit:", round(lower_limit, 2))
print("Upper IQR limit:", round(upper_limit, 2))
print("Potential Fare outliers:", len(fare_outliers))


In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df["Fare"])
plt.title("Fare Before Transformation")
plt.show()


### Outlier decision

The detected high fares are plausible passenger fares rather than obvious data-entry errors. Therefore, removing them would risk losing real observations. We keep them and use `log1p(Fare)` during preprocessing to reduce skewness.


## 3. Feature and Target Identification

The target is `Survived`.

We remove `PassengerId` because it is only an identifier and does not represent a useful passenger characteristic. We also remove `Name` and `Ticket` from the final basic preprocessing because they are high-cardinality text fields and require additional specialized processing.

Useful engineered features will be created from the existing columns.


In [ ]:
target = "Survived"

print("Target variable:", target)
print("Target values:")
print(df[target].value_counts())

X = df.drop(columns=[target])
y = df[target]

print("\nFeature columns:")
print(X.columns.tolist())


## 4. Feature Engineering

We create features that can provide useful information without changing the original observations.

### `FamilySize`
Calculated as `SibSp + Parch + 1`. The passenger is included in the family count.

### `IsAlone`
1 if the passenger travelled alone, otherwise 0.

### `Title`
Extracted from the passenger's name. Titles such as Mr, Mrs, Miss, and Master can provide useful information about age or social role.

### `FarePerPerson`
Fare divided by family size. This gives an approximate fare per family member.

### `AgeGroup`
Creates simple age categories for interpretation. It is an ordinal categorical feature.


In [ ]:
# Family size
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# Whether passenger travelled alone
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# Extract title from name
df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.")[0].str.strip()

# Group uncommon titles
common_titles = ["Mr", "Miss", "Mrs", "Master"]
df["Title"] = df["Title"].where(df["Title"].isin(common_titles), "Other")

# Fare per person
df["FarePerPerson"] = df["Fare"] / df["FamilySize"]

# Age groups
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 35, 60, np.inf],
    labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"],
    include_lowest=True
)

display(df[[
    "Age", "FamilySize", "IsAlone", "Title",
    "Fare", "FarePerPerson", "AgeGroup"
]].head())


## 5. Identify Variable Types

### Numerical variables
`Age`, `Fare`, `FarePerPerson`, `FamilySize`, `SibSp`, `Parch`

### Categorical variables
`Sex`, `Embarked`, `Title`, `AgeGroup`

### Binary variables
`Survived`, `IsAlone`, `CabinKnown`

### Ordinal variable
`Pclass` is ordinal because 1st, 2nd, and 3rd class have an ordered relationship.

### Date/Time variables
There are no date/time variables in this dataset.


## 6. Train-Test Split and Avoiding Data Leakage

To avoid data leakage, the dataset is split into training and validation sets **before fitting preprocessing transformations**.

The median imputer, scaler, and encoder are fitted only on the training data. Scikit-learn's `Pipeline` and `ColumnTransformer` make this process safer.


In [ ]:
from sklearn.model_selection import train_test_split

# Select final features
features = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare",
    "Embarked", "CabinKnown", "FamilySize", "IsAlone",
    "Title", "FarePerPerson", "AgeGroup"
]

X = df[features]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 7. Encoding and Numerical Preprocessing

### Numerical preprocessing
- Median imputation handles any remaining missing numerical values.
- `Fare` and `FarePerPerson` are log-transformed because fares are right-skewed.
- StandardScaler standardizes numerical features.

### Categorical preprocessing
- Missing categorical values are filled using the most frequent category.
- OneHotEncoder converts categorical variables into numerical columns.

`Pclass` is kept as a numerical ordinal feature because passenger classes have a natural order.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

numeric_features = [
    "Age", "SibSp", "Parch", "Fare",
    "FarePerPerson", "FamilySize", "Pclass"
]

categorical_features = [
    "Sex", "Embarked", "Title", "AgeGroup"
]

# Log transform for skewed fare-related variables
def log_transform(X):
    return np.log1p(X)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(log_transform, feature_names_out="one-to-one")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="passthrough"
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)


## 8. Check Skewness Before and After Transformation

The `Fare` variable contains some high values and is strongly right-skewed. A logarithmic transformation reduces the effect of very large values.

This is useful because highly skewed numerical features can make some statistical and machine-learning methods less stable.


In [ ]:
before_skew = X_train["Fare"].skew()

fare_log = np.log1p(X_train["Fare"])
after_skew = fare_log.skew()

print("Fare skewness before log transformation:", round(before_skew, 3))
print("Fare skewness after log transformation:", round(after_skew, 3))

plt.figure(figsize=(8, 4))
sns.histplot(fare_log, kde=True)
plt.title("Log-Transformed Fare")
plt.xlabel("log(1 + Fare)")
plt.show()


## 9. Standardization

Standardization changes numerical variables to a comparable scale using their mean and standard deviation.

This is useful when features have very different numerical ranges, such as `Age`, `Fare`, and `FamilySize`.

The transformation is performed inside the pipeline, and the scaler is fitted only using the training data, which prevents leakage.


In [ ]:
# Show a few processed rows
processed_preview = pd.DataFrame(X_train_processed).head()
display(processed_preview)


## 10. Final Data Quality Check

Before finishing, we check the original cleaned dataframe for missing values and confirm that the engineered features were created successfully.


In [ ]:
print("Missing values in cleaned dataset:")
display(df.isnull().sum().sort_values(ascending=False))

print("\nFinal dataset shape:", df.shape)

print("\nFinal feature list:")
print(features)


## 11. Summary of Cleaning Decisions

| Problem | Method | Reason |
|---|---|---|
| Missing Age | Median imputation | Numerical variable and median is robust to extreme values |
| Missing Embarked | Mode imputation | Categorical variable with very few missing values |
| Missing Cabin | Created `CabinKnown`, then dropped Cabin | Too many missing values in the original column |
| Duplicate rows | Removed with `drop_duplicates()` | Prevent repeated observations |
| Inconsistent text | Stripped spaces and standardized case | Keeps categories consistent |
| Wrong data types | Converted columns to suitable numeric/category types | Makes analysis and preprocessing reliable |
| Fare outliers | Kept valid values and used log transformation | High fares can be genuine, so deleting them could remove useful data |
| Skewed Fare | `log1p()` transformation | Reduces right skew |
| Categorical variables | One-hot encoding | Converts categories to numerical features |
| Numerical variables | Standardization | Puts features on comparable scales |
| Feature engineering | FamilySize, IsAlone, Title, FarePerPerson, AgeGroup | Adds useful information from existing variables |
| Data leakage | Pipeline fitted only on training data | Prevents test information from influencing preprocessing |


## 12. Conclusion

The Titanic dataset was cleaned and prepared for machine-learning analysis. Missing values were handled using median and mode imputation, duplicate records were checked and removed, categorical values were standardized, and inappropriate data types were corrected.

Potential outliers in `Fare` were investigated rather than automatically deleted because high fares can represent genuine first-class passengers. A logarithmic transformation was then used to reduce skewness.

Several useful features were engineered, including `FamilySize`, `IsAlone`, `Title`, `FarePerPerson`, and `AgeGroup`. Categorical variables were one-hot encoded and numerical variables were standardized.

Finally, a train-test split and Scikit-learn preprocessing pipeline were used so that preprocessing was fitted only on training data. This reduces the risk of data leakage and produces a cleaner dataset ready for machine-learning tasks.

### Final Result

The dataset is now cleaned, transformed, encoded, and standardized. The preprocessing workflow can be reused for a classification model that predicts whether a Titanic passenger survived.


## References

1. Kaggle. **Titanic — Machine Learning from Disaster**.  
   https://www.kaggle.com/competitions/titanic

2. Scikit-learn documentation. **Preprocessing and Pipeline**.  
   https://scikit-learn.org/stable/modules/preprocessing.html

### Submission Note

After running all cells successfully:

1. Download this notebook as `.ipynb`.
2. Upload the notebook and relevant dataset/code files to GitHub or Zenodo.
3. Make the repository/file accessible.
4. Submit the GitHub or Zenodo link on LMS.
